# Unified PCS + NPS Hybrid RAG Pipeline

**Architecture:** Staged batch processing with shared preprocessing and metric-specific retrieval/generation

**Stages:**
1. Batch Preprocess all transcripts (shared)
2. Batch Embed all leaf chunks (shared)
3. Conditional RAPTOR for long transcripts (shared)
4. Batch Embed RAPTOR summaries (shared)
5. Metric-specific HyDE Generation + Embedding (PCS & NPS independently)
6. Metric-specific Hybrid Search (PCS & NPS independently)
7. Metric-specific Rationale Generation (PCS & NPS independently)
8. Merge Results + Evaluation

**Key Design:**
- Single input file with both PCS and NPS labeler scores/rationales
- Separate consensus computation per metric
- Shared expensive compute (embedding, RAPTOR) runs once
- Each metric gets unique HyDE queries, retrieval context, and generation prompts
- Separate user-defined rubrics for PCS and NPS
- Supports labeled (calibrated HyDE) and unlabeled (inference HyDE) modes
- Single output table with both PCS and NPS scores/rationales

In [ ]:
# %pip install openai>=1.56.0 FlagEmbedding onnxruntime faiss-cpu numpy pandas scipy scikit-learn nest_asyncio tqdm python-dotenv
# %restart_python

In [ ]:
from openai import AsyncAzureOpenAI
import asyncio
import json
import re
import os
import numpy as np
import pandas as pd
from datetime import datetime
from copy import deepcopy
from collections import Counter
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from tqdm.asyncio import tqdm as atqdm
from tqdm import tqdm
import faiss
import time
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

In [ ]:
load_dotenv()
API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise ValueError("API key not found. Please check your .env file.")

client = AsyncAzureOpenAI(
    api_version="2024-10-21",
    azure_endpoint="https://oai-rtai-dev-aoai-east-001.openai.azure.com/",
    api_key=API_KEY,
)

print("Azure OpenAI client initialized.")

In [ ]:
# --- Pipeline Configuration ---
config = {
    # LLM settings
    "model_name": "gpt-5-mini",
    "reasoning_effort": "low",
    "semaphore_value": 10,
    "n_retries": 30,
    # Hybrid search
    "alpha": 0.7,               # dense vs sparse weight (higher = more dense)
    "top_k": 5,                 # retrieval results count
    # RAPTOR (only for long transcripts)
    "clustering_threshold": 0.3, # agglomerative cosine distance threshold
    # Chunking
    "chunk_token_limit": 500,    # threshold for LLM-based chunking
    # HyDE
    "hyde_mode": "auto",         # auto | calibrated | inference
    # Evaluation
    "eval": True,
    # --- Staged pipeline settings ---
    "token_threshold": 4000,     # transcripts above this use RAPTOR path
    "embed_batch_size": 64,      # BGE-M3 batch size for GPU
    "max_prompt_tokens": 12000,  # max tokens in rationale generation prompt
    "use_fp16": True,            # enable FP16 for A10 GPU
    # --- Rubric paths (user-configurable) ---
    "pcs_rubric_path": "./prompts/PCS_autoprompt1.md",
    "nps_rubric_path": "./prompts/NPS_prompt.md",
    # --- Pipeline toggles ---
    "use_hyde": True,             # enable HyDE hypothesis generation for retrieval queries
    "use_raptor": True,           # enable RAPTOR clustering/summarization for long transcripts
}

In [ ]:
def load_rubric(path: str) -> str:
    """Load a rubric/prompt file from disk."""
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

# Verify rubrics load
pcs_rubric = load_rubric(config['pcs_rubric_path'])
nps_rubric = load_rubric(config['nps_rubric_path'])
print(f"PCS rubric loaded: {len(pcs_rubric)} chars")
print(f"NPS rubric loaded: {len(nps_rubric)} chars")

---
## Data Loading & Dual Consensus Computation

In [ ]:
def compute_consensus(df: pd.DataFrame, metric: str = 'pcs') -> pd.DataFrame:
    """Compute consensus score, class, and rationale for a given metric.

    Args:
        df: DataFrame with wide-format labeler columns
        metric: 'pcs' or 'nps' â€” determines column suffix pattern and class mapping

    Returns:
        DataFrame with added columns: {metric}_consensus_score, {metric}_consensus_class,
        {metric}_consensus_rationale
    """
    score_suffix = f'_{metric}_score'
    rationale_suffix = f'_{metric}_rationale'

    score_cols = [c for c in df.columns if c.endswith(score_suffix)]
    rationale_cols = [c for c in df.columns if c.endswith(rationale_suffix)]

    # Map labeler prefix to their score/rationale columns
    labeler_map = {}
    for col in score_cols:
        prefix = col.replace(score_suffix, '')
        rationale_col = f'{prefix}{rationale_suffix}'
        if rationale_col in rationale_cols:
            labeler_map[prefix] = {'score': col, 'rationale': rationale_col}

    # Metric-specific class mapping
    def score_to_class(score):
        if metric == 'nps':
            if score == -1:
                return 'None'
            elif score <= 6:
                return 'Detractor'
            elif score <= 8:
                return 'Passive'
            else:
                return 'Promoter'
        else:  # pcs
            if score <= 3:
                return 'Negative'
            elif score <= 6:
                return 'Neutral'
            else:
                return 'Positive'

    consensus_scores = []
    consensus_classes = []
    consensus_rationales = []

    for _, row in df.iterrows():
        scores = {}
        for prefix, cols in labeler_map.items():
            val = row[cols['score']]
            if pd.notna(val):
                scores[prefix] = int(val)

        if not scores:
            consensus_scores.append(None)
            consensus_classes.append(None)
            consensus_rationales.append(None)
            continue

        # Majority vote; tie-break with median
        score_values = list(scores.values())
        score_counts = Counter(score_values)
        most_common = score_counts.most_common()

        if len(most_common) == 1 or most_common[0][1] > most_common[1][1]:
            consensus_score = most_common[0][0]
        else:
            consensus_score = int(np.round(np.median(score_values)))

        consensus_class = score_to_class(consensus_score)

        # Rationale from labelers whose score matches consensus
        matching_prefixes = [p for p, s in scores.items() if s == consensus_score]
        if not matching_prefixes:
            matching_prefixes = [p for p, s in scores.items() if abs(s - consensus_score) <= 1]

        rationales = []
        for prefix in matching_prefixes:
            rat = row[labeler_map[prefix]['rationale']]
            if pd.notna(rat) and str(rat).strip():
                rationales.append(str(rat).strip())

        consensus_rationale = ' | '.join(rationales) if rationales else None

        consensus_scores.append(consensus_score)
        consensus_classes.append(consensus_class)
        consensus_rationales.append(consensus_rationale)

    df = df.copy()
    df[f'{metric}_consensus_score'] = consensus_scores
    df[f'{metric}_consensus_class'] = consensus_classes
    df[f'{metric}_consensus_rationale'] = consensus_rationales
    return df


def load_data(path: str = './files/701_nps_pcs_rationales.csv') -> pd.DataFrame:
    """Load combined NPS+PCS dataset and compute consensus for both metrics."""
    df = pd.read_csv(path)
    df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed')], errors='ignore')
    df = df.drop_duplicates(subset='AgentRecordingSessionID').reset_index(drop=True)

    # Compute consensus for both metrics
    df = compute_consensus(df, metric='pcs')
    df = compute_consensus(df, metric='nps')

    output_cols = [
        'AgentRecordingSessionID', 'FormattedText',
        'pcs_consensus_score', 'pcs_consensus_class', 'pcs_consensus_rationale',
        'nps_consensus_score', 'nps_consensus_class', 'nps_consensus_rationale',
    ]
    df = df[output_cols]

    print(f"Loaded {len(df)} transcripts from {path}")
    print(f"  PCS labels: {df['pcs_consensus_score'].notna().sum()}")
    print(f"  NPS labels: {df['nps_consensus_score'].notna().sum()}")
    print(f"  Unlabeled (inference mode): {df['pcs_consensus_score'].isna().sum()}")
    return df

In [ ]:
df = load_data('./files/701_nps_pcs_rationales.csv')
df.head()

---
## Shared Utilities: Preprocessing, Encoding, Search

In [ ]:
# ============================================================
# SHARED UTILITIES: Turn Parsing, Auth/Transfer Detection, Chunking
# ============================================================

TURN_PATTERN = re.compile(r'^(\d+)\s+(Agent|Caller):\s*(.*)$', re.MULTILINE)

def parse_turns(transcript: str) -> list[dict]:
    """Parse transcript into structured turns."""
    turns = []
    for match in TURN_PATTERN.finditer(transcript):
        turns.append({
            'turn_num': int(match.group(1)),
            'speaker': match.group(2),
            'text': match.group(3).strip()
        })
    return turns


AUTH_PATTERNS = [
    r'verify', r'account\s*number', r'name\s+on\s+the\s+account',
    r'last\s+four', r'date\s+of\s+birth', r'address\s+on\s+file',
    r'social\s*security', r'pin\s+number', r'security\s+question',
    r'can\s+i\s+get\s+your\s+name', r'spell\s+that', r'confirm\s+your',
    r'who\s+am\s+i\s+speaking\s+with', r'account\s+holder'
]
AUTH_RE = re.compile('|'.join(AUTH_PATTERNS), re.IGNORECASE)

TRANSFER_PATTERNS = [
    r'transfer', r'supervisor', r'escalat', r'different\s+department',
    r'placing\s+you\s+on\s+hold', r'let\s+me\s+connect\s+you',
    r'specialist', r'another\s+team', r'warm\s+transfer'
]
TRANSFER_RE = re.compile('|'.join(TRANSFER_PATTERNS), re.IGNORECASE)


def detect_auth_section(turns: list[dict], scan_window: int = 12) -> int:
    """Detect end of authentication section in first N turns."""
    last_auth_idx = -1
    window = min(scan_window, len(turns))
    for i in range(window):
        if AUTH_RE.search(turns[i]['text']):
            last_auth_idx = i
    return last_auth_idx + 1 if last_auth_idx >= 0 else 0


def detect_transfer_section(turns: list[dict], scan_window: int = 10) -> int:
    """Detect start of transfer/escalation section from end."""
    first_transfer_idx = len(turns)
    start_scan = max(0, len(turns) - scan_window)
    for i in range(start_scan, len(turns)):
        if TRANSFER_RE.search(turns[i]['text']):
            first_transfer_idx = i
            break
    return first_transfer_idx


def strip_auth_transfer(turns: list[dict]) -> list[dict]:
    """Remove auth opening and transfer/escalation tail."""
    auth_end = detect_auth_section(turns)
    transfer_start = detect_transfer_section(turns)
    core = turns[auth_end:transfer_start]
    return core if len(core) >= 2 else turns


def estimate_tokens(text: str) -> int:
    """Rough token estimate (words * 1.3)."""
    return int(len(text.split()) * 1.3)


def chunk_by_heuristic(turns: list[dict], min_chunk_words: int = 10) -> list[dict]:
    """Simple heuristic chunking: group consecutive turns, merging short turns."""
    if not turns:
        return []
    chunks = []
    current_chunk_turns = [turns[0]]
    for i in range(1, len(turns)):
        turn = turns[i]
        current_text = ' '.join(t['text'] for t in current_chunk_turns)
        if (len(turn['text'].split()) < min_chunk_words or
            len(current_text.split()) < min_chunk_words):
            current_chunk_turns.append(turn)
        else:
            chunks.append(current_chunk_turns)
            current_chunk_turns = [turn]
    if current_chunk_turns:
        chunks.append(current_chunk_turns)

    result = []
    for chunk_turns in chunks:
        text = '\n'.join(f"{t['turn_num']} {t['speaker']}: {t['text']}" for t in chunk_turns)
        result.append({
            'text': text,
            'turn_range': (chunk_turns[0]['turn_num'], chunk_turns[-1]['turn_num']),
            'turns': chunk_turns
        })
    return result


async def chunk_by_llm(turns: list[dict], client: AsyncAzureOpenAI, config: dict) -> list[dict]:
    """Use LLM to segment core conversation into dialogue acts."""
    transcript_text = '\n'.join(
        f"{t['turn_num']} {t['speaker']}: {t['text']}" for t in turns
    )
    prompt = f"""Segment this call transcript into coherent dialogue acts (topic-based sections).
Each segment should represent a distinct phase of the conversation.

Return a JSON array of objects with "start_turn" and "end_turn" (inclusive) for each segment.
Example: [{{"start_turn": 1, "end_turn": 5}}, {{"start_turn": 6, "end_turn": 12}}]

Transcript:
{transcript_text}"""

    try:
        response = await client.chat.completions.create(
            model=config['model_name'],
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            reasoning_effort=config['reasoning_effort'],
            timeout=30.0
        )
        content = response.choices[0].message.content
        parsed = json.loads(content)
        segments = parsed if isinstance(parsed, list) else parsed.get('segments', parsed.get('chunks', []))
        if not segments:
            return chunk_by_heuristic(turns)

        chunks = []
        for seg in segments:
            start, end = seg['start_turn'], seg['end_turn']
            seg_turns = [t for t in turns if start <= t['turn_num'] <= end]
            if seg_turns:
                text = '\n'.join(f"{t['turn_num']} {t['speaker']}: {t['text']}" for t in seg_turns)
                chunks.append({
                    'text': text,
                    'turn_range': (seg_turns[0]['turn_num'], seg_turns[-1]['turn_num']),
                    'turns': seg_turns
                })
        return chunks if chunks else chunk_by_heuristic(turns)
    except Exception as e:
        print(f"LLM chunking failed: {e}, falling back to heuristic")
        return chunk_by_heuristic(turns)


async def preprocess_transcript(transcript: str, client: AsyncAzureOpenAI, config: dict) -> list[dict]:
    """Full preprocessing: parse -> strip -> chunk."""
    turns = parse_turns(transcript)
    if not turns:
        return [{'text': transcript, 'turn_range': (0, 0), 'turns': []}]
    core_turns = strip_auth_transfer(turns)
    total_text = ' '.join(t['text'] for t in core_turns)
    total_tokens = estimate_tokens(total_text)
    if total_tokens < config['chunk_token_limit']:
        return chunk_by_heuristic(core_turns)
    else:
        return await chunk_by_llm(core_turns, client, config)

In [ ]:
# ============================================================
# SHARED UTILITIES: BGE-M3 Encoder
# ============================================================

from FlagEmbedding import BGEM3FlagModel

class BGEM3Encoder:
    """BGE-M3 encoder optimized for batch processing on GPU with FP16."""

    def __init__(self, model_name: str = 'BAAI/bge-m3', use_fp16: bool = True, batch_size: int = 64):
        self.batch_size = batch_size
        print(f"Loading BGE-M3 model: {model_name} (fp16={use_fp16}, batch_size={batch_size})...")
        self.model = BGEM3FlagModel(model_name, use_fp16=use_fp16)
        print("BGE-M3 model loaded.")

    def encode(self, texts: list[str]) -> dict:
        """Encode texts into dense and sparse representations."""
        if not texts:
            return {'dense': np.array([]), 'sparse': []}
        output = self.model.encode(
            texts,
            batch_size=self.batch_size,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
        )
        dense = np.array(output['dense_vecs'])
        sparse_list = []
        for weights in output['lexical_weights']:
            sparse_dict = {int(k) if isinstance(k, str) and k.isdigit() else k: float(v)
                          for k, v in weights.items()}
            sparse_list.append(sparse_dict)
        return {'dense': dense, 'sparse': sparse_list}

In [ ]:
# Initialize encoder (GPU-optimized)
encoder = BGEM3Encoder(use_fp16=config['use_fp16'], batch_size=config['embed_batch_size'])

# Sanity check
demo_enc = encoder.encode(["Test embedding."])
print(f"Embedding dimension: {demo_enc['dense'].shape[1]}")

In [ ]:
# ============================================================
# SHARED UTILITIES: Hybrid Search
# ============================================================

def sparse_dot_product(query_sparse: dict, doc_sparse: dict) -> float:
    score = 0.0
    for token_id, weight in query_sparse.items():
        if token_id in doc_sparse:
            score += weight * doc_sparse[token_id]
    return score


def min_max_normalize(scores: np.ndarray) -> np.ndarray:
    if len(scores) == 0:
        return scores
    min_val, max_val = scores.min(), scores.max()
    if max_val - min_val == 0:
        return np.ones_like(scores)
    return (scores - min_val) / (max_val - min_val)


def hybrid_search(query_dense: np.ndarray, query_sparse: dict, nodes: list[dict], config: dict) -> list[dict]:
    """Hybrid search over nodes (leaf chunks or leaf+summary nodes)."""
    if not nodes:
        return []
    alpha = config['alpha']
    top_k = min(config['top_k'], len(nodes))

    # Dense search with FAISS
    dense_matrix = np.vstack([n['embedding_dense'] for n in nodes]).astype('float32')
    index = faiss.IndexFlatIP(dense_matrix.shape[1])
    index.add(dense_matrix)
    query_vec = query_dense.reshape(1, -1).astype('float32')
    dense_scores_all, dense_indices = index.search(query_vec, len(nodes))
    dense_scores_all, dense_indices = dense_scores_all[0], dense_indices[0]

    dense_scores = np.zeros(len(nodes))
    for idx, score in zip(dense_indices, dense_scores_all):
        if idx >= 0:
            dense_scores[idx] = score

    # Sparse scoring
    sparse_scores = np.array([sparse_dot_product(query_sparse, n['embedding_sparse']) for n in nodes])

    # Normalize and fuse
    dense_norm = min_max_normalize(dense_scores)
    sparse_norm = min_max_normalize(sparse_scores)
    fused_scores = alpha * dense_norm + (1 - alpha) * sparse_norm

    top_indices = np.argsort(fused_scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        node = nodes[idx]
        results.append({
            'text': node['text'],
            'turn_range': node['turn_range'],
            'level': node.get('level', 0),
            'node_type': node.get('node_type', 'leaf'),
            'score': float(fused_scores[idx]),
            'dense_score': float(dense_norm[idx]),
            'sparse_score': float(sparse_norm[idx]),
        })
    return results

---
## Stage 1: Batch Preprocess All Transcripts (Shared)

In [ ]:
async def stage_1_batch_preprocess(df: pd.DataFrame, client: AsyncAzureOpenAI, config: dict) -> dict:
    """Preprocess all transcripts concurrently.

    Returns dict keyed by AgentRecordingSessionID -> {
        'chunks': list[dict], 'token_count': int,
        'is_long': bool, 'preprocessed_text': str
    }
    """
    print(f"\n{'='*60}")
    print(f"STAGE 1: Batch Preprocessing ({len(df)} transcripts)")
    print(f"{'='*60}")

    start_time = time.time()
    sem = asyncio.Semaphore(config['semaphore_value'])
    token_threshold = config['token_threshold']

    async def preprocess_one(row):
        async with sem:
            arsid = row['AgentRecordingSessionID']
            transcript = row['FormattedText']
            chunks = await preprocess_transcript(transcript, client, config)
            preprocessed_text = '\n\n'.join(c['text'] for c in chunks)
            token_count = estimate_tokens(preprocessed_text)
            is_long = token_count > token_threshold
            return arsid, {
                'chunks': chunks,
                'token_count': token_count,
                'is_long': is_long,
                'preprocessed_text': preprocessed_text,
            }

    tasks = [preprocess_one(row) for _, row in df.iterrows()]
    results = await atqdm.gather(*tasks, desc="Preprocessing")

    transcript_data = {arsid: data for arsid, data in results}

    elapsed = time.time() - start_time
    total_chunks = sum(len(d['chunks']) for d in transcript_data.values())
    n_long = sum(1 for d in transcript_data.values() if d['is_long'])
    n_short = len(transcript_data) - n_long

    print(f"\n  Completed in {elapsed:.1f}s")
    print(f"  Total transcripts: {len(transcript_data)}")
    print(f"  Total chunks: {total_chunks}")
    print(f"  Avg chunks/transcript: {total_chunks / len(transcript_data):.1f}")
    print(f"  Short path (<=4000 tokens): {n_short}")
    print(f"  Long/RAPTOR path (>4000 tokens): {n_long}")

    return transcript_data

---
## Stage 2: Batch Embed All Leaf Chunks (Shared)

In [ ]:
def stage_2_batch_embed_chunks(transcript_data: dict, encoder: BGEM3Encoder) -> dict:
    """Embed all chunks from all transcripts in a single batch GPU call."""
    print(f"\n{'='*60}")
    print(f"STAGE 2: Batch Embed All Leaf Chunks")
    print(f"{'='*60}")

    start_time = time.time()

    # Flatten all chunk texts with index mapping
    all_texts = []
    index_map = []  # (arsid, chunk_idx)
    for arsid, data in transcript_data.items():
        for i, chunk in enumerate(data['chunks']):
            all_texts.append(chunk['text'])
            index_map.append((arsid, i))

    print(f"  Total texts to embed: {len(all_texts)}")

    # Single batch encode
    embeddings = encoder.encode(all_texts)

    # Distribute back
    for flat_idx, (arsid, chunk_idx) in enumerate(index_map):
        chunk = transcript_data[arsid]['chunks'][chunk_idx]
        chunk['embedding_dense'] = embeddings['dense'][flat_idx]
        chunk['embedding_sparse'] = embeddings['sparse'][flat_idx]
        chunk['level'] = 0
        chunk['node_type'] = 'leaf'

    elapsed = time.time() - start_time
    throughput = len(all_texts) / elapsed if elapsed > 0 else 0

    print(f"  Embedded {len(all_texts)} texts in {elapsed:.1f}s")
    print(f"  Throughput: {throughput:.1f} texts/sec")
    print(f"  Dense shape: ({len(all_texts)}, {embeddings['dense'].shape[1]})")

    return transcript_data

---
## Stage 3: Conditional RAPTOR for Long Transcripts (Shared)

In [ ]:
async def summarize_cluster(texts: list[str], client: AsyncAzureOpenAI, config: dict) -> str:
    """Generate abstractive summary of a cluster of transcript chunks."""
    combined = '\n---\n'.join(texts)
    prompt = f"""Summarize the following section(s) of a customer service call transcript.
Focus on: agent behavior, professionalism, problem resolution, and customer sentiment.
Write a concise 2-3 sentence summary.

{combined}"""
    for attempt in range(config['n_retries']):
        try:
            response = await client.chat.completions.create(
                model=config['model_name'],
                messages=[{"role": "user", "content": prompt}],
                reasoning_effort=config['reasoning_effort'],
                timeout=30.0
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt == config['n_retries'] - 1:
                return f"Summary unavailable: {combined[:200]}"
            await asyncio.sleep(1)


async def stage_3_raptor_clustering(transcript_data: dict, client: AsyncAzureOpenAI, config: dict) -> dict:
    """Run RAPTOR clustering + LLM summarization for long transcripts only."""
    print(f"\n{'='*60}")
    print(f"STAGE 3: Conditional RAPTOR (Long Transcripts Only)")
    print(f"{'='*60}")

    start_time = time.time()
    long_arsids = [arsid for arsid, data in transcript_data.items() if data['is_long']]

    if not long_arsids:
        print(f"  No long transcripts found. Skipping RAPTOR stage.")
        return transcript_data

    print(f"  Processing {len(long_arsids)} long transcripts...")

    sem = asyncio.Semaphore(config['semaphore_value'])
    threshold = config['clustering_threshold']

    cluster_tasks = []
    for arsid in long_arsids:
        chunks = transcript_data[arsid]['chunks']
        if len(chunks) <= 3:
            cluster_tasks.append((arsid, [c['text'] for c in chunks], 'single'))
        else:
            dense_matrix = np.vstack([c['embedding_dense'] for c in chunks])
            distances = pdist(dense_matrix, metric='cosine')
            Z = linkage(distances, method='average')
            cluster_labels = fcluster(Z, t=threshold, criterion='distance')
            clusters = {}
            for idx, label in enumerate(cluster_labels):
                clusters.setdefault(label, []).append(idx)
            for cluster_id, member_indices in clusters.items():
                cluster_texts = [chunks[i]['text'] for i in member_indices]
                cluster_tasks.append((arsid, cluster_texts, f'cluster_{cluster_id}'))

    print(f"  Total cluster summaries to generate: {len(cluster_tasks)}")

    async def summarize_one(arsid, cluster_texts, cluster_id):
        async with sem:
            summary = await summarize_cluster(cluster_texts, client, config)
            return arsid, cluster_id, summary

    tasks = [summarize_one(a, t, c) for a, t, c in cluster_tasks]
    results = await atqdm.gather(*tasks, desc="RAPTOR Summaries")

    summaries_by_arsid = {}
    for arsid, cluster_id, summary in results:
        summaries_by_arsid.setdefault(arsid, []).append(summary)

    # Build root summaries
    root_tasks = []
    for arsid, summaries in summaries_by_arsid.items():
        transcript_data[arsid]['raptor_summaries'] = summaries
        if len(summaries) > 1:
            root_tasks.append((arsid, summaries))
        else:
            transcript_data[arsid]['root_summary'] = summaries[0]

    if root_tasks:
        async def root_summarize(arsid, summaries):
            async with sem:
                root = await summarize_cluster(summaries, client, config)
                return arsid, root
        root_results = await atqdm.gather(
            *[root_summarize(a, s) for a, s in root_tasks], desc="Root Summaries"
        )
        for arsid, root_summary in root_results:
            transcript_data[arsid]['root_summary'] = root_summary

    elapsed = time.time() - start_time
    total_summaries = sum(len(s) for s in summaries_by_arsid.values())
    print(f"\n  Completed in {elapsed:.1f}s")
    print(f"  Long transcripts processed: {len(long_arsids)}")
    print(f"  Total cluster summaries: {total_summaries}")
    print(f"  Root summaries generated: {len(root_tasks)}")

    return transcript_data

---
## Stage 4: Batch Embed RAPTOR Summaries (Shared)

In [ ]:
def stage_4_batch_embed_raptor(transcript_data: dict, encoder: BGEM3Encoder) -> dict:
    """Batch embed all RAPTOR summary texts in one GPU call."""
    print(f"\n{'='*60}")
    print(f"STAGE 4: Batch Embed RAPTOR Summaries")
    print(f"{'='*60}")

    start_time = time.time()

    all_texts = []
    index_map = []
    for arsid, data in transcript_data.items():
        if not data.get('is_long') or 'raptor_summaries' not in data:
            continue
        for i, summary_text in enumerate(data['raptor_summaries']):
            all_texts.append(summary_text)
            index_map.append((arsid, 'summary', i))
        if 'root_summary' in data:
            all_texts.append(data['root_summary'])
            index_map.append((arsid, 'root', 0))

    if not all_texts:
        print(f"  No RAPTOR summaries to embed. Skipping.")
        return transcript_data

    print(f"  Total summary texts to embed: {len(all_texts)}")
    embeddings = encoder.encode(all_texts)

    # Init raptor_nodes
    for arsid, data in transcript_data.items():
        if data.get('is_long') and 'raptor_summaries' in data:
            data['raptor_nodes'] = []
            data['root_node'] = None

    # Distribute
    for flat_idx, (arsid, node_type, local_idx) in enumerate(index_map):
        dense_vec = embeddings['dense'][flat_idx]
        sparse_vec = embeddings['sparse'][flat_idx]
        if node_type == 'summary':
            text = transcript_data[arsid]['raptor_summaries'][local_idx]
            transcript_data[arsid]['raptor_nodes'].append({
                'text': text, 'turn_range': (0, 0),
                'embedding_dense': dense_vec, 'embedding_sparse': sparse_vec,
                'level': 1, 'node_type': 'summary'
            })
        elif node_type == 'root':
            text = transcript_data[arsid]['root_summary']
            transcript_data[arsid]['root_node'] = {
                'text': text, 'turn_range': (0, 0),
                'embedding_dense': dense_vec, 'embedding_sparse': sparse_vec,
                'level': 2, 'node_type': 'root'
            }

    elapsed = time.time() - start_time
    throughput = len(all_texts) / elapsed if elapsed > 0 else 0
    print(f"  Embedded {len(all_texts)} summaries in {elapsed:.1f}s")
    print(f"  Throughput: {throughput:.1f} texts/sec")

    return transcript_data

---
## Stage 5: Metric-Specific HyDE Generation + Embedding (PCS / NPS)

In [ ]:
def stage_5_alt_embed_query(transcript_data: dict, encoder: BGEM3Encoder, metric: str = 'pcs') -> dict:
    """Alternative to Stage 5 when HyDE is disabled.

    Uses truncated preprocessed text as the retrieval query instead of LLM-generated hypotheses.
    BGE-M3 embeds the text directly for hybrid search.
    """
    print(f"\n{'='*60}")
    print(f"STAGE 5-ALT: Direct Query Embedding (HyDE OFF) [{metric.upper()}]")
    print(f"{'='*60}")

    start_time = time.time()

    # Collect query texts (truncated preprocessed text)
    arsid_order = list(transcript_data.keys())
    query_texts = []
    for arsid in arsid_order:
        data = transcript_data[arsid]
        query_text = data['preprocessed_text'][:1500]
        query_texts.append(query_text)
        data[f'hyde_text_{metric}'] = query_text
        data[f'hyde_mode_{metric}'] = 'disabled'

    print(f"  Embedding {len(query_texts)} query texts [{metric.upper()}]...")

    # Batch encode with BGE-M3
    embeddings = encoder.encode(query_texts)

    for i, arsid in enumerate(arsid_order):
        transcript_data[arsid][f'hyde_dense_{metric}'] = embeddings['dense'][i]
        transcript_data[arsid][f'hyde_sparse_{metric}'] = embeddings['sparse'][i]

    elapsed = time.time() - start_time
    print(f"  Completed in {elapsed:.1f}s (no LLM calls)")
    print(f"  HyDE disabled \u2014 using preprocessed text as query")

    return transcript_data

In [ ]:
HYDE_PCS_INFERENCE_PROMPT = """You are evaluating a customer service call for Post-Call Sentiment (PCS).
Based on the PCS evaluation criteria and the call context below, generate a hypothetical
rationale that describes what evidence of agent performance (positive or negative) might
look like in this call. Focus on specific behaviors: empathy, resolution, professionalism,
communication clarity, ownership, and customer effort.

PCS Evaluation Criteria:
{rubric}

Call Context:
{context}

Generate a 2-4 sentence hypothetical rationale describing the agent's performance evidence:"""


HYDE_NPS_INFERENCE_PROMPT = """You are evaluating a customer service call for Net Promoter Score (NPS).
Based on the NPS evaluation criteria and the call context below, generate a hypothetical
rationale that describes what evidence of customer brand sentiment (positive or negative)
might look like in this call. Focus on: customer satisfaction signals, frustration indicators,
brand loyalty cues, service experience quality, and likelihood to recommend.

NPS Evaluation Criteria:
{rubric}

Call Context:
{context}

Generate a 2-4 sentence hypothetical rationale describing the customer's brand sentiment evidence:"""


async def stage_5_hyde_generation(
    transcript_data: dict, df: pd.DataFrame,
    encoder: BGEM3Encoder, client: AsyncAzureOpenAI, config: dict,
    metric: str = 'pcs'
) -> dict:
    """Generate HyDE hypotheses for all transcripts for a specific metric, then batch-embed.

    Args:
        metric: 'pcs' or 'nps' â€” determines rubric, inference prompt, and calibration source.

    Mutates transcript_data: adds 'hyde_dense_{metric}', 'hyde_sparse_{metric}',
    'hyde_text_{metric}', 'hyde_mode_{metric}'.
    """
    print(f"\n{'='*60}")
    print(f"STAGE 5: HyDE Generation + Embedding [{metric.upper()}]")
    print(f"{'='*60}")

    start_time = time.time()
    sem = asyncio.Semaphore(config['semaphore_value'])
    hyde_mode = config['hyde_mode']

    # Load metric-specific rubric
    rubric_path = config[f'{metric}_rubric_path']
    rubric = load_rubric(rubric_path)[:1500]

    # Select inference prompt template
    inference_prompt = HYDE_PCS_INFERENCE_PROMPT if metric == 'pcs' else HYDE_NPS_INFERENCE_PROMPT

    # Rationale lookup (metric-specific consensus rationale)
    rationale_col = f'{metric}_consensus_rationale'
    rationale_lookup = dict(zip(df['AgentRecordingSessionID'], df[rationale_col]))

    calibrated_arsids = []
    inference_arsids = []

    for arsid in transcript_data.keys():
        rationale = rationale_lookup.get(arsid)
        has_rationale = pd.notna(rationale) and str(rationale).strip() if rationale else False

        use_calibrated = False
        if hyde_mode == 'calibrated' and has_rationale:
            use_calibrated = True
        elif hyde_mode == 'auto' and has_rationale:
            use_calibrated = True

        if use_calibrated:
            transcript_data[arsid][f'hyde_text_{metric}'] = str(rationale).strip()
            transcript_data[arsid][f'hyde_mode_{metric}'] = 'calibrated'
            calibrated_arsids.append(arsid)
        else:
            inference_arsids.append(arsid)

    print(f"  Calibrated (no LLM): {len(calibrated_arsids)}")
    print(f"  Inference (LLM needed): {len(inference_arsids)}")

    # Generate hypotheses for inference-mode transcripts
    if inference_arsids:
        async def generate_hyde_one(arsid):
            async with sem:
                data = transcript_data[arsid]
                if data['is_long'] and 'root_summary' in data:
                    context = data['root_summary']
                else:
                    context = data['preprocessed_text'][:1500]
                prompt = inference_prompt.format(rubric=rubric, context=context)
                try:
                    response = await client.chat.completions.create(
                        model=config['model_name'],
                        messages=[{"role": "user", "content": prompt}],
                        reasoning_effort=config['reasoning_effort'],
                        timeout=30.0
                    )
                    hypothesis = response.choices[0].message.content.strip()
                except Exception as e:
                    hypothesis = data['preprocessed_text'][:500]
                return arsid, hypothesis

        inference_results = await atqdm.gather(
            *[generate_hyde_one(arsid) for arsid in inference_arsids],
            desc=f"HyDE Inference [{metric.upper()}]"
        )
        for arsid, hypothesis in inference_results:
            transcript_data[arsid][f'hyde_text_{metric}'] = hypothesis
            transcript_data[arsid][f'hyde_mode_{metric}'] = 'inference'

    llm_time = time.time() - start_time

    # Batch embed all HyDE queries for this metric
    print(f"  Embedding all HyDE queries [{metric.upper()}]...")
    embed_start = time.time()
    arsid_order = list(transcript_data.keys())
    all_hyde_texts = [transcript_data[arsid][f'hyde_text_{metric}'] for arsid in arsid_order]
    hyde_embeddings = encoder.encode(all_hyde_texts)

    for i, arsid in enumerate(arsid_order):
        transcript_data[arsid][f'hyde_dense_{metric}'] = hyde_embeddings['dense'][i]
        transcript_data[arsid][f'hyde_sparse_{metric}'] = hyde_embeddings['sparse'][i]

    embed_time = time.time() - embed_start
    total_time = time.time() - start_time
    print(f"\n  LLM generation time: {llm_time:.1f}s")
    print(f"  Embedding time: {embed_time:.1f}s")
    print(f"  Total Stage 5 [{metric.upper()}] time: {total_time:.1f}s")

    return transcript_data

---
## Stage 6: Metric-Specific Hybrid Search (PCS / NPS)

In [ ]:
def stage_6_hybrid_search(transcript_data: dict, config: dict, metric: str = 'pcs') -> dict:
    """Run hybrid search for each transcript using metric-specific HyDE embeddings.

    Short: search leaf chunks only. Long: search leaves + RAPTOR nodes.
    Mutates transcript_data: adds 'retrieved_passages_{metric}'.
    """
    print(f"\n{'='*60}")
    print(f"STAGE 6: Hybrid Search [{metric.upper()}]")
    print(f"{'='*60}")

    start_time = time.time()

    for arsid, data in tqdm(transcript_data.items(), desc=f"Search [{metric.upper()}]"):
        nodes = list(data['chunks'])
        if data['is_long']:
            if 'raptor_nodes' in data and data['raptor_nodes']:
                nodes.extend(data['raptor_nodes'])
            if 'root_node' in data and data['root_node']:
                nodes.append(data['root_node'])

        retrieved = hybrid_search(
            query_dense=data[f'hyde_dense_{metric}'],
            query_sparse=data[f'hyde_sparse_{metric}'],
            nodes=nodes,
            config=config
        )
        data[f'retrieved_passages_{metric}'] = retrieved

    elapsed = time.time() - start_time
    avg_time = elapsed / len(transcript_data) if transcript_data else 0

    all_scores = [p['score'] for d in transcript_data.values() for p in d[f'retrieved_passages_{metric}']]
    node_types = [p['node_type'] for d in transcript_data.values() for p in d[f'retrieved_passages_{metric}']]
    type_counts = Counter(node_types)

    print(f"\n  Completed in {elapsed:.1f}s")
    print(f"  Avg time per transcript: {avg_time*1000:.1f}ms")
    if all_scores:
        top1_scores = [d[f'retrieved_passages_{metric}'][0]['score']
                       for d in transcript_data.values() if d[f'retrieved_passages_{metric}']]
        print(f"  Avg top-1 score: {np.mean(top1_scores):.3f}")
    print(f"  Retrieved node types: {dict(type_counts)}")

    return transcript_data

---
## Stage 7: Metric-Specific Rationale Generation (PCS / NPS)

In [ ]:
def get_output_schema(metric: str = 'pcs') -> dict:
    """Get the structured output schema for a given metric."""
    if metric == 'nps':
        schema_definition = {
            "name": "nps_evaluation",
            "description": "Net Promoter Score evaluation with citation-grounded rationale.",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "class": {"type": "string", "enum": ["Promoter", "Passive", "Detractor", "None"]},
                    "score": {"type": "integer", "description": "NPS score from -1 to 10."},
                    "rationale": {"type": "string", "description": "Rationale with inline citations."}
                },
                "required": ["class", "score", "rationale"],
                "additionalProperties": False
            }
        }
    else:  # pcs
        schema_definition = {
            "name": "pcs_evaluation",
            "description": "Post-Call Sentiment evaluation with citation-grounded rationale.",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "class": {"type": "string", "enum": ["Negative", "Neutral", "Positive"]},
                    "score": {"type": "integer", "description": "PCS score from 0 to 10."},
                    "rationale": {"type": "string", "description": "Rationale with inline citations."}
                },
                "required": ["class", "score", "rationale"],
                "additionalProperties": False
            }
        }
    return {"type": "json_schema", "json_schema": schema_definition}


GENERATION_PROMPT_SHORT = """You are evaluating a customer service call.

## Full Transcript
{full_transcript}

## Key Evidence Passages (Most Relevant to Evaluation)
The following passages were identified as the most relevant evidence:

{retrieved_passages}

## Instructions
Based on the evaluation rubric and the full transcript above (with key evidence highlighted):
1. Assign the appropriate class and score per the rubric
2. Write a rationale that includes inline quoted citations from the transcript
3. Format citations as: ('exact quoted text from the transcript')
4. Ground every claim in your rationale with at least one citation

Evaluate thoroughly using the provided rubric."""


GENERATION_PROMPT_LONG = """You are evaluating a customer service call.

## Call Summary
{root_summary}

## Key Evidence Passages (Most Relevant to Evaluation)
The following passages were retrieved as the most relevant evidence from the transcript:

{retrieved_passages}

## Transcript Context (Truncated)
{truncated_transcript}

## Instructions
Based on the evaluation rubric, call summary, and evidence above:
1. Assign the appropriate class and score per the rubric
2. Write a rationale that includes inline quoted citations from the transcript
3. Format citations as: ('exact quoted text from the transcript')
4. Ground every claim in your rationale with at least one citation

Evaluate thoroughly using the provided rubric."""


def build_rationale_prompt(data: dict, config: dict, metric: str = 'pcs') -> str:
    """Build the appropriate rationale prompt based on transcript length.

    Falls back to SHORT prompt if RAPTOR is disabled (no root_summary available).
    """
    passages_text = ''
    for i, p in enumerate(data[f'retrieved_passages_{metric}']):
        passages_text += f"\n### Passage {i+1} (Turns {p['turn_range'][0]}-{p['turn_range'][1]}, {p['node_type']}):\n"
        passages_text += p['text'] + '\n'

    if data['is_long'] and 'root_summary' in data:
        # Long transcript WITH RAPTOR summaries available
        root_summary = data['root_summary']
        max_tokens = config['max_prompt_tokens']
        used_tokens = estimate_tokens(root_summary) + estimate_tokens(passages_text) + 300
        remaining_budget = max(500, max_tokens - used_tokens)

        full_text = data['preprocessed_text']
        words = full_text.split()
        max_words = int(remaining_budget / 1.3)
        if len(words) > max_words:
            truncated = ' '.join(words[:max_words]) + '\n\n[... transcript truncated ...]'
        else:
            truncated = full_text

        return GENERATION_PROMPT_LONG.format(
            root_summary=root_summary,
            retrieved_passages=passages_text,
            truncated_transcript=truncated
        )
    else:
        # Short transcript OR long transcript without RAPTOR (fallback)
        max_tokens = config['max_prompt_tokens']
        full_text = data['preprocessed_text']
        used_tokens = estimate_tokens(passages_text) + 300
        remaining_budget = max(500, max_tokens - used_tokens)
        words = full_text.split()
        max_words = int(remaining_budget / 1.3)
        if len(words) > max_words:
            full_text = ' '.join(words[:max_words]) + '\n\n[... transcript truncated ...]'

        return GENERATION_PROMPT_SHORT.format(
            full_transcript=full_text,
            retrieved_passages=passages_text
        )


async def stage_7_batch_rationale(
    transcript_data: dict, client: AsyncAzureOpenAI, config: dict,
    metric: str = 'pcs'
) -> dict:
    """Generate rationale evaluations for all transcripts for a specific metric.

    Mutates transcript_data: adds 'result_{metric}' dict with class, score, rationale.
    """
    print(f"\n{'='*60}")
    print(f"STAGE 7: Rationale Generation [{metric.upper()}]")
    print(f"{'='*60}")

    start_time = time.time()
    sem = asyncio.Semaphore(config['semaphore_value'])

    # Load metric-specific rubric as system prompt
    system_prompt = load_rubric(config[f'{metric}_rubric_path'])
    response_format = get_output_schema(metric)

    async def generate_one(arsid: str, data: dict):
        async with sem:
            user_prompt = build_rationale_prompt(data, config, metric)
            for attempt in range(config['n_retries']):
                try:
                    response = await client.chat.completions.with_raw_response.create(
                        model=config['model_name'],
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt}
                        ],
                        response_format=response_format,
                        reasoning_effort=config['reasoning_effort'],
                        timeout=60.0
                    )
                    headers = response.headers
                    parsed = response.parse()
                    content = json.loads(parsed.choices[0].message.content)
                    debug_info = {
                        'prompt_tokens': parsed.usage.prompt_tokens,
                        'completion_tokens': parsed.usage.completion_tokens,
                        'total_tokens': parsed.usage.total_tokens,
                        'cached_tokens': getattr(parsed.usage.prompt_tokens_details, 'cached_tokens', 0),
                        'model': parsed.model,
                        'processing_ms': headers.get('openai-processing-ms'),
                    }
                    return arsid, {
                        'class': content['class'], 'score': content['score'],
                        'rationale': content['rationale'],
                        'debug_info': debug_info, 'error': None
                    }
                except Exception as e:
                    if attempt == config['n_retries'] - 1:
                        return arsid, {
                            'class': 'error', 'score': -1,
                            'rationale': f'Generation failed: {str(e)}',
                            'debug_info': {'error': str(e)}, 'error': str(e)
                        }
                    await asyncio.sleep(1)

    tasks = [generate_one(arsid, data) for arsid, data in transcript_data.items()]
    results = await atqdm.gather(*tasks, desc=f"Rationale [{metric.upper()}]")

    total_tokens = 0
    errors = 0
    for arsid, result in results:
        transcript_data[arsid][f'result_{metric}'] = result
        if result['error']:
            errors += 1
        elif 'total_tokens' in result.get('debug_info', {}):
            total_tokens += result['debug_info']['total_tokens']

    elapsed = time.time() - start_time
    print(f"\n  Completed in {elapsed:.1f}s")
    print(f"  Successful: {len(transcript_data) - errors}")
    print(f"  Errors: {errors}")
    print(f"  Total tokens used: {total_tokens:,}")
    print(f"  Avg tokens/call: {total_tokens // max(1, len(transcript_data) - errors):,}")

    return transcript_data

---
## Stage 8: Unified Pipeline Driver & Evaluation

In [ ]:
async def unified_pipeline_driver(
    df: pd.DataFrame, config: dict,
    encoder: BGEM3Encoder, client: AsyncAzureOpenAI,
    use_hyde: bool = None, use_raptor: bool = None
) -> pd.DataFrame:
    """Run the full unified Hybrid RAG pipeline for both PCS and NPS.

    Args:
        use_hyde: Override config['use_hyde']. When False, uses preprocessed text as query.
        use_raptor: Override config['use_raptor']. When False, skips Stages 3-4.
    """
    # Resolve toggles: function args override config
    use_hyde = use_hyde if use_hyde is not None else config.get('use_hyde', True)
    use_raptor = use_raptor if use_raptor is not None else config.get('use_raptor', True)

    pipeline_start = time.time()
    stage_times = {}

    print(f"\n{'#'*60}")
    print(f"# UNIFIED PCS + NPS HYBRID RAG PIPELINE")
    print(f"# Transcripts: {len(df)} | Token threshold: {config['token_threshold']}")
    print(f"# Embed batch size: {config['embed_batch_size']} | FP16: {config['use_fp16']}")
    print(f"# HyDE: {'ON' if use_hyde else 'OFF'} | RAPTOR: {'ON' if use_raptor else 'OFF'}")
    print(f"# PCS rubric: {config['pcs_rubric_path']}")
    print(f"# NPS rubric: {config['nps_rubric_path']}")
    print(f"{'#'*60}")

    # ===== SHARED STAGES 1-4 =====

    # Stage 1: Batch Preprocess
    t0 = time.time()
    transcript_data = await stage_1_batch_preprocess(df, client, config)
    stage_times['1_preprocess'] = time.time() - t0

    # Stage 2: Batch Embed Chunks
    t0 = time.time()
    transcript_data = stage_2_batch_embed_chunks(transcript_data, encoder)
    stage_times['2_embed_chunks'] = time.time() - t0

    # Stage 3: Conditional RAPTOR
    if use_raptor:
        t0 = time.time()
        transcript_data = await stage_3_raptor_clustering(transcript_data, client, config)
        stage_times['3_raptor'] = time.time() - t0

        # Stage 4: Embed RAPTOR Summaries
        t0 = time.time()
        transcript_data = stage_4_batch_embed_raptor(transcript_data, encoder)
        stage_times['4_embed_raptor'] = time.time() - t0
    else:
        print(f"\n{'='*60}")
        print(f"RAPTOR disabled \u2014 skipping Stages 3-4")
        print(f"{'='*60}")
        stage_times['3_raptor'] = 0.0
        stage_times['4_embed_raptor'] = 0.0

    # ===== METRIC-SPECIFIC STAGES 5-7 (PCS) =====

    if use_hyde:
        # Stage 5a: PCS HyDE
        t0 = time.time()
        transcript_data = await stage_5_hyde_generation(transcript_data, df, encoder, client, config, metric='pcs')
        stage_times['5a_hyde_pcs'] = time.time() - t0
    else:
        # Stage 5a-alt: Embed preprocessed text as query (no LLM)
        t0 = time.time()
        transcript_data = stage_5_alt_embed_query(transcript_data, encoder, metric='pcs')
        stage_times['5a_hyde_pcs'] = time.time() - t0

    # Stage 6a: PCS Hybrid Search
    t0 = time.time()
    transcript_data = stage_6_hybrid_search(transcript_data, config, metric='pcs')
    stage_times['6a_search_pcs'] = time.time() - t0

    # Stage 7a: PCS Rationale Generation
    t0 = time.time()
    transcript_data = await stage_7_batch_rationale(transcript_data, client, config, metric='pcs')
    stage_times['7a_rationale_pcs'] = time.time() - t0

    # ===== METRIC-SPECIFIC STAGES 5-7 (NPS) =====

    if use_hyde:
        # Stage 5b: NPS HyDE
        t0 = time.time()
        transcript_data = await stage_5_hyde_generation(transcript_data, df, encoder, client, config, metric='nps')
        stage_times['5b_hyde_nps'] = time.time() - t0
    else:
        # Stage 5b-alt: Embed preprocessed text as query (no LLM)
        t0 = time.time()
        transcript_data = stage_5_alt_embed_query(transcript_data, encoder, metric='nps')
        stage_times['5b_hyde_nps'] = time.time() - t0

    # Stage 6b: NPS Hybrid Search
    t0 = time.time()
    transcript_data = stage_6_hybrid_search(transcript_data, config, metric='nps')
    stage_times['6b_search_nps'] = time.time() - t0

    # Stage 7b: NPS Rationale Generation
    t0 = time.time()
    transcript_data = await stage_7_batch_rationale(transcript_data, client, config, metric='nps')
    stage_times['7b_rationale_nps'] = time.time() - t0

    # ===== STAGE 8: BUILD RESULTS =====
    total_time = time.time() - pipeline_start

    # Build consensus lookup from input df
    consensus_cols = ['pcs_consensus_score', 'pcs_consensus_rationale',
                      'nps_consensus_score', 'nps_consensus_rationale']
    available_consensus_cols = [c for c in consensus_cols if c in df.columns]
    if available_consensus_cols:
        consensus_lookup = df.set_index('AgentRecordingSessionID')[available_consensus_cols].to_dict('index')
    else:
        consensus_lookup = {}

    rows = []
    for arsid, data in transcript_data.items():
        pcs_result = data.get('result_pcs', {})
        nps_result = data.get('result_nps', {})

        n_tree_nodes = len(data['chunks'])
        if data['is_long']:
            n_tree_nodes += len(data.get('raptor_nodes', []))
            if data.get('root_node'):
                n_tree_nodes += 1

        consensus = consensus_lookup.get(arsid, {})

        rows.append({
            'AgentRecordingSessionID': arsid,
            # Consensus ground truth
            'pcs_consensus_score': consensus.get('pcs_consensus_score'),
            'pcs_consensus_rationale': consensus.get('pcs_consensus_rationale'),
            'nps_consensus_score': consensus.get('nps_consensus_score'),
            'nps_consensus_rationale': consensus.get('nps_consensus_rationale'),
            # PCS results
            'pcs_score': pcs_result.get('score', -1),
            'pcs_class': pcs_result.get('class', 'error'),
            'pcs_rationale': pcs_result.get('rationale', ''),
            # NPS results
            'nps_score': nps_result.get('score', -1),
            'nps_class': nps_result.get('class', 'error'),
            'nps_rationale': nps_result.get('rationale', ''),
            # Pipeline metadata
            'hyde_mode_pcs': data.get('hyde_mode_pcs', 'unknown'),
            'hyde_mode_nps': data.get('hyde_mode_nps', 'unknown'),
            'n_chunks': len(data['chunks']),
            'n_tree_nodes': n_tree_nodes,
            'is_long': data['is_long'],
            'token_count': data['token_count'],
            'processing_time_s': round(total_time / len(transcript_data), 2),
            # Errors
            'pcs_error': pcs_result.get('error'),
            'nps_error': nps_result.get('error'),
        })

    results_df = pd.DataFrame(rows)

    # Save
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = f'./outputs/{timestamp}'
    os.makedirs(output_dir, exist_ok=True)
    results_df.to_csv(f'{output_dir}/unified_nps_pcs_results.csv', index=False)

    # Report
    print(f"\n\n{'#'*60}")
    print(f"# PIPELINE COMPLETE")
    print(f"{'#'*60}")
    print(f"\n  Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"  Throughput: {len(df) / (total_time / 3600):.0f} transcripts/hr")
    print(f"  Toggles: HyDE={'ON' if use_hyde else 'OFF'}, RAPTOR={'ON' if use_raptor else 'OFF'}")
    print(f"\n  Per-stage timing:")
    for stage, t in stage_times.items():
        pct = (t / total_time) * 100
        print(f"    {stage}: {t:.1f}s ({pct:.0f}%)")

    pcs_errors = results_df[results_df['pcs_error'].notna()]
    nps_errors = results_df[results_df['nps_error'].notna()]
    print(f"\n  Results:")
    print(f"    Total transcripts: {len(results_df)}")
    print(f"    PCS successful: {len(results_df) - len(pcs_errors)} | errors: {len(pcs_errors)}")
    print(f"    NPS successful: {len(results_df) - len(nps_errors)} | errors: {len(nps_errors)}")
    print(f"    Output: {output_dir}/unified_nps_pcs_results.csv")

    return results_df

In [ ]:
def evaluate_results(results_df: pd.DataFrame, df_labels: pd.DataFrame, config: dict) -> dict:
    """Evaluate pipeline results against consensus labels for both PCS and NPS."""

    print(f"\n{'='*60}")
    print(f"EVALUATION RESULTS")
    print(f"{'='*60}")

    metrics_report = {}

    for metric in ['pcs', 'nps']:
        score_col = f'{metric}_consensus_score'
        class_col = f'{metric}_consensus_class'
        pred_score_col = f'{metric}_score'
        pred_class_col = f'{metric}_class'
        error_col = f'{metric}_error'

        # Merge predictions with labels
        eval_df = results_df.merge(
            df_labels[['AgentRecordingSessionID', score_col, class_col]],
            on='AgentRecordingSessionID', how='inner'
        )

        # Filter to valid rows
        eval_df = eval_df[
            (eval_df[pred_class_col] != 'error') & (eval_df[score_col].notna())
        ].copy()

        if eval_df.empty:
            print(f"\n  [{metric.upper()}] No valid evaluation pairs found.")
            continue

        eval_df[pred_score_col] = pd.to_numeric(eval_df[pred_score_col], errors='coerce')
        eval_df[score_col] = pd.to_numeric(eval_df[score_col], errors='coerce')
        eval_df = eval_df.dropna(subset=[pred_score_col, score_col])

        # Compute metrics
        eval_df[f'{metric}_correct'] = (
            (eval_df[pred_score_col] - eval_df[score_col]).abs() <= 1.5
        ).astype(int)
        eval_df[f'{metric}_class_match'] = (
            eval_df[pred_class_col] == eval_df[class_col]
        ).astype(int)

        score_accuracy = eval_df[f'{metric}_correct'].mean()
        class_accuracy = eval_df[f'{metric}_class_match'].mean()
        mae = (eval_df[pred_score_col] - eval_df[score_col]).abs().mean()

        print(f"\n  [{metric.upper()}] (n={len(eval_df)})")
        print(f"    Score accuracy (Â±1.5): {score_accuracy:.3f}")
        print(f"    Class accuracy:        {class_accuracy:.3f}")
        print(f"    Mean absolute error:   {mae:.2f}")

        # Per-class breakdown
        per_class = (
            eval_df.groupby(class_col)
            .agg(
                count=(class_col, 'count'),
                score_accuracy=(f'{metric}_correct', 'mean'),
                class_match_rate=(f'{metric}_class_match', 'mean'),
                avg_pred_score=(pred_score_col, 'mean'),
                avg_true_score=(score_col, 'mean'),
            )
            .reset_index()
        )
        print(f"\n    Per-Class Breakdown:")
        print(f"    {per_class.to_string(index=False)}")

        metrics_report[metric] = {
            'n': len(eval_df),
            'score_accuracy': score_accuracy,
            'class_accuracy': class_accuracy,
            'mae': mae,
            'per_class': per_class,
        }

    # Summary comparison
    if 'pcs' in metrics_report and 'nps' in metrics_report:
        print(f"\n  {'='*50}")
        print(f"  SIDE-BY-SIDE COMPARISON")
        print(f"  {'='*50}")
        print(f"  {'Metric':<20} {'PCS':<15} {'NPS':<15}")
        print(f"  {'-'*50}")
        print(f"  {'Score Acc (Â±1.5)':<20} {metrics_report['pcs']['score_accuracy']:<15.3f} {metrics_report['nps']['score_accuracy']:<15.3f}")
        print(f"  {'Class Accuracy':<20} {metrics_report['pcs']['class_accuracy']:<15.3f} {metrics_report['nps']['class_accuracy']:<15.3f}")
        print(f"  {'MAE':<20} {metrics_report['pcs']['mae']:<15.2f} {metrics_report['nps']['mae']:<15.2f}")

    print(f"\n  Config: alpha={config['alpha']}, top_k={config['top_k']}, token_threshold={config['token_threshold']}")
    print(f"    hyde_mode={config['hyde_mode']}, embed_batch_size={config['embed_batch_size']}")

    return metrics_report

---
## Run Pipeline

In [ ]:
# --- Demo: small batch (5 transcripts) ---
demo_df = df.head(5).copy()
demo_results = asyncio.run(unified_pipeline_driver(demo_df, config, encoder, client))

display_cols = ['AgentRecordingSessionID', 'pcs_consensus_score', 'pcs_score', 'pcs_class',
                'nps_consensus_score', 'nps_score', 'nps_class',
                'hyde_mode_pcs', 'hyde_mode_nps', 'is_long', 'n_chunks', 'token_count']
print(demo_results[display_cols].to_string(index=False))

print(f"\nSample PCS rationale:")
print(demo_results.iloc[0]['pcs_rationale'][:500])
print(f"\nSample NPS rationale:")
print(demo_results.iloc[0]['nps_rationale'][:500])

In [ ]:
# --- Full Evaluation Run (labeled transcripts only) ---
# Filter to transcripts with at least one metric labeled
labeled_df = df[
    df['pcs_consensus_score'].notna() | df['nps_consensus_score'].notna()
].copy()
print(f"Running unified pipeline on {len(labeled_df)} labeled transcripts...")

eval_results_df = asyncio.run(unified_pipeline_driver(labeled_df, config, encoder, client))
metrics_report = evaluate_results(eval_results_df, df, config)

In [ ]:
# --- Full Dataset Run (including unlabeled / inference mode) ---
# Uncomment to run:
# all_results_df = asyncio.run(unified_pipeline_driver(df, config, encoder, client))
# metrics_report = evaluate_results(all_results_df, df, config)

In [ ]:
# --- Config Tuning ---
# Adjust parameters and re-run:
# config['alpha'] = 0.5
# config['top_k'] = 7
# config['token_threshold'] = 3000
# config['hyde_mode'] = 'inference'
# config['pcs_rubric_path'] = './prompts/PCS_autoprompt.md'  # swap rubric

# eval_results_df = asyncio.run(unified_pipeline_driver(labeled_df, config, encoder, client))
# metrics_report = evaluate_results(eval_results_df, df, config)

In [ ]:
# --- Toggle Validation: All Combinations ---
# Run a small batch with all four toggle combinations to verify correctness.
# Uncomment to run:

# demo_df = df.head(5).copy()
# toggle_configs = [
#     {"use_hyde": True, "use_raptor": True, "label": "HyDE=ON, RAPTOR=ON (baseline)"},
#     {"use_hyde": False, "use_raptor": True, "label": "HyDE=OFF, RAPTOR=ON"},
#     {"use_hyde": True, "use_raptor": False, "label": "HyDE=ON, RAPTOR=OFF"},
#     {"use_hyde": False, "use_raptor": False, "label": "HyDE=OFF, RAPTOR=OFF"},
# ]
#
# toggle_results = {}
# for tc in toggle_configs:
#     print(f"\n{'*'*60}")
#     print(f"* RUNNING: {tc['label']}")
#     print(f"{'*'*60}")
#     result_df = asyncio.run(unified_pipeline_driver(
#         demo_df, config, encoder, client,
#         use_hyde=tc['use_hyde'], use_raptor=tc['use_raptor']
#     ))
#     toggle_results[tc['label']] = result_df
#     print(f"  -> Completed. Rows: {len(result_df)}, "
#           f"PCS errors: {result_df['pcs_error'].notna().sum()}, "
#           f"NPS errors: {result_df['nps_error'].notna().sum()}")
#
# # Compare scores across configurations
# print(f"\n\n{'='*60}")
# print("TOGGLE COMPARISON")
# print(f"{'='*60}")
# for label, rdf in toggle_results.items():
#     print(f"\n{label}:")
#     print(f"  PCS scores: {rdf['pcs_score'].tolist()}")
#     print(f"  NPS scores: {rdf['nps_score'].tolist()}")
#     print(f"  HyDE modes (PCS): {rdf['hyde_mode_pcs'].tolist()}")
#     print(f"  Consensus PCS: {rdf['pcs_consensus_score'].tolist()}")
#     print(f"  Consensus NPS: {rdf['nps_consensus_score'].tolist()}")